# # Scenario 2 — 170 km / 186 GBaud: performance vs complexity

Performance-vs-complexity figure: received SNR (peak over power) vs computational
complexity [RM/2D] for EDC, Ideal DBP, OSSFM, LDBP, ESSFM and L-ESSFM, at the two
oversamplings n=1.125 (solid) and n=2 (dashed).

NEW scenario (not yet trained). Set up training (see tutorial) to fill
`results/`, then this reproduces the figure. 4x bandwidth -> larger NLPR filters.


In [ ]:
import sys, os
_ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, os.path.join(_ROOT, 'core')); os.chdir(_ROOT)
import numpy as np
import matplotlib.pyplot as plt
from system import build_system
from backprop import curve
from complexity import eval_compl, complexity_ldbp

SCENARIO = 'scenario_170km_186GBd'
CONFIG   = f'config/{SCENARIO}.ini'
RESDIR   = f'results/{SCENARIO}'
L_KM, R_GBD = 170e3, 186e9

## 1. Load the forward (received signal)
Uses the cached forward for this scenario. (For the exercise we reuse the shared
forward; regenerate with `core/forward.py` for a fully independent one.)


In [ ]:
# Dario's shared forward (dict {P_W:[(y,x)]}); convert to {Pdbm:[(y,x)]}
fwd = np.load('/home/dario/ldbp2/forward_resultsMAX.npy', allow_pickle=True).item()
data = {}
for Pw, reals in fwd.items():
    data[round(10*np.log10(Pw/1e-3), 4)] = [(np.array(y), np.array(x)) for (y, x) in reals]
Pgrid = np.array(sorted(data))
S = build_system(CONFIG)
print(f'{len(Pgrid)} powers, forward loaded')


## 2. Reference curves: EDC and Ideal DBP
Deterministic; computed directly on the forward.


In [ ]:
edc  = curve(S, data, Pgrid, 'edc', 1, edc=True).max()
idl  = curve(S, data, Pgrid, 'ideal', 800).max()
print(f'EDC peak = {edc:.3f} dB | Ideal DBP peak = {idl:.3f} dB')


## 3. L-ESSFM and ESSFM curves (load trained models)
SNR peak at each Ns from the shipped `parameters.csv`.  ESSFM = tied NLPR + rho.


In [ ]:
def peak_for(tag, ns):
    p = f'{RESDIR}/{tag}_Ns{ns}/parameters.csv'
    if not os.path.exists(p):
        return None
    return curve(S, data, Pgrid, 'lessfm', ns, p).max()

NS_LESSFM = [1,2,3,4,5,6,7,8,9,10,20,30,40,50]
lessfm = {ns: peak_for('lessfm', ns) for ns in NS_LESSFM}
lessfm = {k:v for k,v in lessfm.items() if v is not None}
print('L-ESSFM:', {k: round(v,3) for k,v in lessfm.items()})
# ESSFM models (if shipped); else this stays empty
NS_ESSFM = [1,2,3,5,10,20,30,40,50]
essfm = {ns: peak_for('essfm', ns) for ns in NS_ESSFM}
essfm = {k:v for k,v in essfm.items() if v is not None}
print('ESSFM:', {k: round(v,3) for k,v in essfm.items()})


## 4. Assemble the performance-vs-complexity figure

In [ ]:
nos = S['OS_d']  # 1.125 for this scenario
plt.figure(figsize=(9,6))
# L-ESSFM / ESSFM share the frequency-domain complexity eval_compl(Ns)
if lessfm:
    ns = sorted(lessfm)
    plt.plot([eval_compl(n, nos, L_KM, R_GBD) for n in ns], [lessfm[n] for n in ns],
             'b-o', label='L-ESSFM')
if essfm:
    ns = sorted(essfm)
    plt.plot([eval_compl(n, nos, L_KM, R_GBD) for n in ns], [essfm[n] for n in ns],
             'm-x', label='ESSFM')
plt.axhline(idl, color='r', label='Ideal DBP')
plt.axhline(edc, color='y', label='EDC')
plt.xscale('log'); plt.xlim(1e1, None)
plt.xlabel('Complexity $C_M$ [RM/2D]'); plt.ylabel('Received SNR [dB]')
plt.title(f'{SCENARIO}: SNR vs complexity (n={nos})'); plt.grid(True, alpha=0.3); plt.legend()
plt.show()


## 5. Notes
- OSSFM and LDBP curves can be added the same way (OSSFM via `curve(..., 'ossfm', ns)`
  with grid-searched scale; LDBP via `core/ldbp.py` + `complexity_ldbp`).
- For n=2 set the n=2 scenario config / models and overlay as dashed lines.
- To retrain from scratch set up a short-block training config and call
  `core/lessfm.py` (see the tutorial notebook).
